# vrgrid — the T4 columnThe three measurements from `scripts/aws/t4.sh run` that need a **second, larger** GPUthan the laptop RTX 5050 (8 GB). Everything else in the T4 pass is already done on thelaptop and lives in `docs/gpu-lane/laptop/`.1. `timing_table.py` — the frame-loop column beside the laptop's **22.3 / 26.7 ms**2. `gpu_parity.py` — CPU/CUDA map hashes must match on a **different card and driver**3. `vram_contention.py` — grid-vs-FRNet contention on **16 GB** instead of 8**Settings → Accelerator: GPU T4 x2 · Internet: On.** Both need a phone-verified account.**Attach two datasets:** `parthibhang/semantickitti` (the 96 GB mirror) and the`vrgrid-bundle` upload (official KITTI GT poses + the FRNet checkpoint — the twothings the mirror does not carry).

In [ ]:
# 1. The machine. This is the whole point of the notebook -- a DIFFERENT card.!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csvimport torchprint("torch", torch.__version__, "| cuda", torch.version.cuda)assert torch.cuda.is_available(), "no GPU -- set Accelerator to GPU in the sidebar"name = torch.cuda.get_device_name(0)vram = torch.cuda.get_device_properties(0).total_memory / 1e9print(f"{name}, {vram:.1f} GB")assert vram > 12, f"{name} has {vram:.1f} GB -- the 16 GB contention test needs more than the laptop's 8"

In [ ]:
# 2. The repo. Public, so no token.%cd /kaggle/working!rm -rf vrgrid-26!git clone -q https://github.com/Stxtics03/vrgrid-26.git%cd /kaggle/working/vrgrid-26!git log --oneline -1!pip -q install -e . 2>&1 | tail -2try:    import cupy; print("cupy", cupy.__version__, "(preinstalled)")except ImportError:    !pip -q install cupy-cuda12x

In [ ]:
# 3. Patchwork++ is optional -- gpu_parity takes --no-patchworkpp if the build fails.import subprocessHAVE_PW = Falsetry:    import pypatchworkpp; HAVE_PW = Trueexcept ImportError:    r = subprocess.run("git clone -q --depth 1 https://github.com/url-kaist/patchwork-plusplus.git /tmp/pw "                       "&& pip -q install /tmp/pw/python", shell=True, capture_output=True, text=True)    try:        import pypatchworkpp; HAVE_PW = True    except ImportError:        print("Patchwork++ unavailable; gpu_parity will run with --no-patchworkpp")print("patchwork++:", HAVE_PW)

In [ ]:
# 4. Assemble the asset tree vrgrid expects. The mirror supplies sequences/; the#    bundle supplies poses/ and the checkpoint.#    loader.py uses the OFFICIAL KITTI GT poses at poses/<seq>.txt, NOT the#    SemanticKITTI SLAM poses inside sequences/<seq>/poses.txt. Do not substitute one#    for the other -- they are different quantities and the swap is silent.import os, globfrom pathlib import PathMIRROR = Path("/kaggle/input/semantickitti")BUNDLE = next(iter(glob.glob("/kaggle/input/*/poses")), None)assert BUNDLE, "vrgrid-bundle not attached -- add it under Input in the sidebar"BUNDLE = Path(BUNDLE).parentA = Path("/kaggle/working/assets")(A / "dataset").mkdir(parents=True, exist_ok=True)for src, dst in [(MIRROR / "sequences", A / "dataset/sequences"),                 (BUNDLE / "poses",     A / "dataset/poses"),                 (BUNDLE / "checkpoints", A / "checkpoints")]:    assert src.exists(), f"missing: {src}"    if not dst.exists():        dst.symlink_to(src)os.environ["VRGRID_ASSETS"] = str(A)os.environ["VRGRID_DATA_ROOT"] = str(A / "dataset")os.environ["VRGRID_FRNET_CHECKPOINT"] = str(A / "checkpoints/frnet-semantickitti_seg.pth")for k in ("VRGRID_ASSETS", "VRGRID_DATA_ROOT", "VRGRID_FRNET_CHECKPOINT"):    print(f"{k}={os.environ[k]}")

In [ ]:
# 5. Prove the mirror is the real sequence 08 before any number is produced.#    These md5s were taken from the laptop's own copy. A third-party mirror is#    data, not a promise -- if a byte differs, the T4 column is not comparable#    to the laptop column and nothing below is worth running.import hashlibfrom pathlib import PathREF = { "velodyne/000000.bin": "ec4dac4b5ca2c6b6980364dfa12ba5cb", "labels/000000.label": "8c3f4ea72be55eb61147a3cfc0eb371f", "velodyne/000100.bin": "7de939b8ce0ced796be0460d1079b9dc", "labels/000100.label": "9785a6b37f8db8ab75a205782c51ee61", "velodyne/004070.bin": "0a15440a9b9b23512449e8c4d7152f7a", "labels/004070.label": "d4235c8a5c3ad6b8c4979899c176b283",}S8 = Path(os.environ["VRGRID_DATA_ROOT"]) / "sequences/08"n_bin = len(list((S8 / "velodyne").glob("*.bin")))n_lbl = len(list((S8 / "labels").glob("*.label")))print(f"seq 08: {n_bin} scans, {n_lbl} labels (expect 4071 / 4071)")assert (n_bin, n_lbl) == (4071, 4071), "sequence 08 is incomplete in this mirror"bad = []for rel, want in REF.items():    got = hashlib.md5((S8 / rel).read_bytes()).hexdigest()    ok = got == want    print(f"  {'OK  ' if ok else 'FAIL'} {rel}  {got}")    if not ok: bad.append(rel)assert not bad, f"mirror differs from the laptop copy at {bad} -- do not use these numbers"print("\nmirror matches the laptop's sequence 08 byte for byte")

In [ ]:
# 6. The three measurements.import pathlibOUT = pathlib.Path("/kaggle/working/t4"); OUT.mkdir(exist_ok=True)!nvidia-smi > {OUT}/host.log 2>&1!nvcc --version >> {OUT}/host.log 2>&1!lscpu | head -20 >> {OUT}/host.log 2>&1!python scripts/timing_table.py --seq 08 --frames 200 --device cpu  2>&1 | tee {OUT}/timing_cpu.log

In [ ]:
!python scripts/timing_table.py --seq 08 --frames 200 --device cuda 2>&1 | tee {OUT}/timing_cuda.log

In [ ]:
PW = "" if HAVE_PW else "--no-patchworkpp"!python scripts/gpu_parity.py --seq 08 --frames 200 {PW} 2>&1 | tee {OUT}/parity.log

In [ ]:
# The one that actually needed 16 GB: on the laptop's 8 GB the one-loop# configuration was the one that missed 10 Hz.!python scripts/vram_contention.py --seq 08 --frames 200 --pair-seconds 40 \        --out {OUT}/vram-contention.json 2>&1 | tee {OUT}/vram_contention.log

In [ ]:
# 7. Collect. Download /kaggle/working/t4-results.zip and drop the contents into#    docs/gpu-lane/t4/ in the repo.!cd /kaggle/working && zip -qr t4-results.zip t4 && ls -la t4/ && echo && du -h t4-results.zip